# Simplifed versions of subplots for XA, XN/XAU, and XQ+XR/XU/XAA/XAB/XAG/XAM

These are used to produce the node positionings, topology, and total number of descendant samples used to create Fig 2. The total number of hidden sample descendants of each node is printed using the `num_sample_desc()` function (note that the number shown underneath a collapsed node in the normal subgraph visualization is the number of immediate children).

For subgraphs of __all__ the Pango X lineages, not just the 3 in Fig 2, see supp_pdf-subgraphs-PangoX.ipynb 

In [ ]:
import sc2ts
import nb_utils
import numpy as np
from IPython.display import HTML
import tszip


ts = tszip.load("../data/sc2ts_viridian_v2.2.trees.tsz")

# Join with the associated data
df = sc2ts.node_data(ts).set_index("sample_id")

# Load in the ARG to the visualizer - can take a few minutes
arg = nb_utils.D3ARG_viz(ts, df, pangolin_field='pango')

In [ ]:
arg.set_sc2ts_node_labels(add_strain_names=False)
arg.d3arg.nodes.loc[arg.d3arg.nodes.id == 94150, 'label'] = "Alpha\nroot"
arg.d3arg.nodes.loc[arg.d3arg.nodes.id == 167136, 'label'] = "Delta\nroot"
arg.d3arg.nodes.loc[arg.d3arg.nodes.id == 1149197, 'label'] = "BA.1\nroot"
arg.d3arg.nodes.loc[arg.d3arg.nodes.id == 1171662, 'label'] = "BA.2\nroot"
arg.d3arg.nodes.loc[arg.d3arg.nodes.id == 871718, 'label'] = "BA.3\nroot"
arg.d3arg.nodes.loc[arg.d3arg.nodes.id == 1519017, 'label'] = "BA.4\nroot"
arg.d3arg.nodes.loc[arg.d3arg.nodes.id == 1428716, 'label'] = "BA.5\nroot"
arg.set_sc2ts_node_styles()

In [ ]:
import collections
import tskit

def output_layout_positions(arg_info, prefix, colours):
    # Output saved positions and format details to a file that can be read in by matplotlib to make modifiable
    # subgraph plots for nicer figures (e.g. Fig 2)
    used_node_ids = arg_info.included.nodes
    used_nodes = arg.d3arg.nodes.loc[used_node_ids]
    used_nodes.sort_values("time", inplace=True)
    excluded = set()
    node_data = ["id,type,color,label_inside,label_outside,outside_pos,x,y"]
    edge_data = ["id,parent,child,style,color,num_mutations"]
    muts = ts.mutations_edge
    for u, row in used_nodes.iterrows():
        inside_label = None
        outside_pos = "NA"
        is_re = (row.ts_flags & sc2ts.NODE_IS_RECOMBINANT) != 0
        is_sample = (row.ts_flags & tskit.NODE_IS_SAMPLE) != 0
        node = ts.node(u)
        pango = node.metadata.get("pango")
        parent_edges = np.where(ts.edges_child == u)[0]
        child_edges = np.where(ts.edges_parent == u)[0]
        if is_sample and len(pango) <= 4:
            inside_label = pango
        if inside_label is None:
            if is_re or np.min(node.time - ts.nodes_time[ts.edges_child[child_edges]]) < 5:
                outside_pos = "top"
            else:
                outside_pos = "right"
                
        node_data.append(",".join([
            str(u),                                                              #ID
            "recombinant" if is_re else "sample" if is_sample else "internal",   #Type 
            "black" if is_re else colours.get(pango, "white" if is_sample else "grey"), #Colour
            inside_label or "NA",                                                #Label_inside
            pango if inside_label is None else "NA",                             #Label_outside
            outside_pos,
            str(row.x_pos_01),
            str(row.time),
        ]))
        for edge in parent_edges:
            if ts.edges_parent[edge] in used_node_ids:
                edge_data.append(",".join([
                    str(len(edge_data)),
                    str(ts.edges_parent[edge]),
                    str(u),
                    "solid",
                    "black",
                    str(int(np.sum(muts==edge))),
                ]))
            else:
                # Anchor the roots
                node_data.append(",".join([
                    str(ts.edges_parent[edge]), #ID
                    "anchor",                   #Type 
                    "transparent",              #Colour
                    "NA",                       #Label_inside
                    "NA",                       #Label_outside
                    "NA",
                    str(row.x_pos_01),
                    str(row.time),
                ]))
                edge_data.append(",".join([
                    str(len(edge_data)),
                    str(ts.edges_parent[edge]),
                    str(u),
                    "dashdot",
                    "black",
                    str(0),
                ]))
        # Now check on descendants
        # Hack here by taking the first and last tree
        excluded |= {u}
        samples = {s for s in ts.first().samples(u)}
        samples |= {s for s in ts.last().samples(u)}
        unique_desc = samples - excluded
        pangos = {ts.node(s).metadata.get("pango") for s in unique_desc}
        if len(unique_desc) > 0:
            node_data.append(",".join([
                str(-u),
                "clade",
                "magenta" if len({colours.get(p) for p in pangos}) > 1 else colours.get(pangos.pop(), "white"),
                str(len(unique_desc)),
                "NA",
                "NA",
                str(row.x_pos_01),
                str(row.time),  # This will be adjusted by the plotting code
            ]))
            edge_data.append(",".join([
                str(len(edge_data)),
                str(u),
                str(-u),
                "dashed",
                "black",
                str(0),
            ]))
        excluded |= samples

    with open(f"{prefix}_nodes.csv", "wt") as fn:
        print("\n".join(node_data), file=fn)
    with open(f"{prefix}_edges.csv", "wt") as fe:
        print("\n".join(edge_data), file=fe)

In [ ]:
colours = {"XA": "yellow", "XN": "#D1605E", "XAU": "#85B5B2"}

In [ ]:
# Exclude all XAs apart from a select few
dfXA = df[np.logical_and(df.pango == "XA", df.is_sample)]
exclude = dfXA.node_id[np.isin(dfXA.index, ["ERR5308556", "ERR5414941", "ERR5651144"]) == False].values

arg_info = arg.plot_pango_subgraph(
    "XA",
    height=500,
    exclude=exclude,
    y_axis_scale="time",
    oldest_y_label="2021-01",
    child_levels=0,
    parent_levels=2,
    positions_file="layout_data/XA-min.json",
    highlight_nodes={colours["XA"]: dfXA.node_id},
)
output_layout_positions(arg_info, "layout_data/XA_base", colours)

In [ ]:
dfXN = df[np.logical_and(df.pango == "XN", df.is_sample)]
dfXAU = df[np.logical_and(df.pango == "XAU", df.is_sample)]

keepXN_XAU = ["SRR20786472", "ERR9626571", "ERR9627436", "ERR9827586"]
inclBA2 = df.loc[[
    "ERR12781112", "ERR12963523", "ERR12965588", "ERR13126948", "ERR9794253", "ERR9969775",
    # "ERR9502469", "ERR9584988", "ERR9794107", "ERR9760073", "ERR9714811", "SRR21890741", "ERR12754078", "ERR12964085", #BA.2 root
    "SRR20777279", "ERR12964964", 

], 'node_id']

exclude = list(dfXN.node_id[np.isin(dfXN.index, keepXN_XAU) == False])
exclude += list(dfXAU.node_id[np.isin(dfXAU.index, keepXN_XAU) == False])

arg_info = arg.plot_pango_subgraph(
    ["XN", "XAU"],
    height=500,
    exclude=exclude,
    include=list(inclBA2) + [3406100],
    y_axis_scale="time",
    oldest_y_label="2022-01",
    child_levels=0,
    parent_levels=3,
    positions_file="layout_data/XN-XAU-min.json",
    highlight_nodes={colours["XN"]: dfXN.node_id, colours["XAU"]: dfXAU.node_id},
)
output_layout_positions(arg_info, "layout_data/XN-XAU_base", colours)

In [ ]:
col = ['#332288', '#88CCEE', '#44AA99', '#999933', '#DDCC77', '#882255', '#CC6677']  # "Muted" from https://sronpersonalpages.nl/~pault/
pangoX = ["XAA", "XAB", "XAG", "XAM", "XQ", "XR", "XU"]

keep_ids = list(df.loc[[
    "ERR8691075", "ERR9471138", "ERR8992833", "ERR9262366", "ERR9368764", "ERR9401688", # XQ (58 samples)
    "ERR9086279", "SRR20570640", "ERR9394584", "ERR9470478", "ERR12960015", # XR (18) + one nested BA.2
    "ERR12958672", "ERR9663506", "SRR20778325",
    "SRR18665648", "ERR9208650", 
    "ERR9790459", # XU (1)
    "SRR19088714", "SRR18887001", # XAA (22)
    "ERR12965629", "ERR11863985", # XAB (3)
    "ERR13172793", # XAG (17)
    "ERR13172793", "ERR12969122", "SRR20123998", # XAM (26)
], 'node_id'])


cmap = {}
for c, pX in zip(col, pangoX):
    df_tmp = df[np.logical_and(df.pango == pX, df.is_sample)]
    cmap[c] = list(df_tmp.node_id)
    colours[pX] = c

arg_info = arg.plot_sc2ts_subgraph(
    keep_ids,
    y_axis_scale="time",
    height=600,
    width=1200,
    parent_levels=2,
    highlight_colour='#DDDDDD',
    child_levels=0,
    highlight_nodes=cmap,
    positions_file="layout_data/XAA-XAB-XAG-XAM-XQ-XR-XU-min.json",
    oldest_y_label="2021-10",
)

output_layout_positions(
    arg_info,
    "layout_data/XAA-XAB-XAG-XAM-XQ-XR-XU_base",
    {k: v for k, v in colours.items() if k in pangoX},  # only send in the colours in this plot, so we don't count XA in descendants etc.
)

In [ ]:
## VISUAL CHECK ON THE FULL SUBGRAPH


keep_ids = list(df.loc[["SRR19689888", "ERR8146303", "ERR8163061", "SRR19689888", "ERR9099799"], 'node_id'])

cmap = {}
for c, pX in zip(col, pangoX):
    df_tmp = df[np.logical_and(df.pango == pX, df.is_sample)]
    exclude += list(df_tmp.node_id[np.isin(df_tmp.index, keep) == False])
    cmap[c] = list(df_tmp.node_id)
    colours[pX] = c

arg_info = arg.plot_pango_subgraph(
    pangoX,
    y_axis_scale="rank",
    height=1200,
    width=1200,
    include=keep_ids,
    parent_levels=2,
    child_levels=0,
    highlight_nodes=cmap,
    oldest_y_label="2021-09",
    positions_file="layout_data/XAA-XAB-XAG-XAM-XQ-XR-XU-min.json",
)
